# EM Fixed Income Analytics Suite

#### Programming choices:

Start date = 2015-01-01, capturing multiple EM stress situations

List of problems faced:

- Data Sourcing: WRDS didn't work, despite successfully conecting my account it does not have the necessary info, had to manually download
- File names in Investing.com sometimes had typos -> quick fix by renaming, this even bypassed the column check

In [1]:
import pandas as pd
import glob
import re
from pathlib import Path
from collections import defaultdict

## 1. Yield Curve PCA & Regime Detection

### Downloading Data

#### Country Selection

In [2]:
local_list_of_countries = ['Brazil','Mexico','South Africa','Poland']
hard_list_of_countries = ['Colombia', 'Hungary','Romania']
all_countries = sorted(set(local_list_of_countries + hard_list_of_countries))

#### Data Scraping

Getting all the data for countries via Investing.com
Maturities vary per country: maximizing coverage, avoiding redudancy between adjacent points when possible

As I don't have access to my Bloomberg while doing this project, I'll need to find another way to obtain Hard currency bond yields.
Proxy spreads: local-currency yields - US treasury yields

Source: https://www.investing.com/rates-bonds/

Jordan was removed from the hard currency list due to limited data availability from Investing.com

#### Cleaning the Data

In [3]:
base_dir = Path(r'data\raw')
all_unique_columns = set()
column_provenance = defaultdict(list)

for file_path in base_dir.rglob("*.csv"):
    try:
        raw_columns = pd.read_csv(file_path, nrows=0).columns.tolist()
        all_unique_columns.update(raw_columns)

        for col in raw_columns:
            column_provenance[col].append(file_path.name)
    except Exception as e:
        print(f"Failed to read header of {file_path.name}: {e}")

print(f"Total Unique Raw Columns Discovered: {len(all_unique_columns)}\n")
for col in sorted(all_unique_columns):
    file_count = len(column_provenance[col])
    print(f"'{col}' -> Found in {file_count} files")

Total Unique Raw Columns Discovered: 6

'Change %' -> Found in 35 files
'Date' -> Found in 35 files
'High' -> Found in 35 files
'Low' -> Found in 35 files
'Open' -> Found in 35 files
'Price' -> Found in 35 files


In [4]:
files = glob.glob("data/raw/*/*.csv")
print("Files found:", len(files))

country_files = defaultdict(list)
for f in files:
    match = re.search(r"(\w[\w\s]+?)\s+(\d+)-Year Bond", f)
    if match:
        country  = match.group(1)
        maturity = match.group(2) + "Y"
        country_files[country].append((maturity, f))
    else:
        print(f"NO MATCH: {f}")

country_dfs = {}
for country, maturity_files in country_files.items():
    dfs = []
    for maturity, fname in maturity_files:
        df = pd.read_csv(fname, usecols=["Date", "Price"])
        df["Date"] = pd.to_datetime(df["Date"])
        df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
        df = df.rename(columns={"Price": maturity})
        df = df.set_index("Date")
        dfs.append(df)
    
    combined = pd.concat(dfs, axis=1).sort_index().loc["2015-01-01":"2026-05-01"]
    combined = combined[sorted(combined.columns, key=lambda x: int(x[:-1]))]
    country_dfs[country] = combined

for country, df in country_dfs.items():
    print(f"{country}: {df.shape} | columns: {list(df.columns)}")

Files found: 35
Brazil: (3016, 5) | columns: ['2Y', '3Y', '5Y', '8Y', '10Y']
Colombia: (3403, 5) | columns: ['2Y', '4Y', '5Y', '10Y', '15Y']
Hungary: (2962, 5) | columns: ['1Y', '5Y', '10Y', '15Y', '20Y']
Mexico: (3260, 5) | columns: ['3Y', '5Y', '10Y', '20Y', '30Y']
Poland: (3457, 5) | columns: ['1Y', '2Y', '3Y', '5Y', '10Y']
Romania: (3452, 5) | columns: ['1Y', '3Y', '5Y', '7Y', '10Y']
South Africa: (2891, 5) | columns: ['5Y', '10Y', '12Y', '20Y', '30Y']


In [9]:
for country, df in country_dfs.items():
    df.to_csv(f"data/output/{country}.csv")
    print(f"Saved {country}")

Saved Brazil
Saved Colombia
Saved Hungary
Saved Mexico
Saved Poland
Saved Romania
Saved South Africa


### PCA